In [5]:
import pandas as pd
import numpy as np
import feature_selection as fs
import optuna
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import matthews_corrcoef
from catboost import CatBoostClassifier
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from ydata_profiling import ProfileReport
import getting_threshold_genes as gtg
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score
from ydata_profiling import ProfileReport
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import matthews_corrcoef
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import model as m
import catboost_model as miaw

from sklearn.metrics import make_scorer, matthews_corrcoef
from sklearn.model_selection import cross_val_score
from sklearn.svm import SVC

Ok, we will use a SVM

In [2]:
eigth_bins_original = pd.read_csv("/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/Results/master/SMOTE/RNAs_z_Combat_test.csv")
eigth_bins_original=eigth_bins_original[eigth_bins_original["Experiment"]=="GSE167186"]
seq_metadata_path = "/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/Data/All_rna_samples_metadata_edit.csv"
seq_metadata = pd.read_csv(seq_metadata_path)
rnaSeq_origial= pd.merge(eigth_bins_original, seq_metadata, on='Sample', how='inner')
rnaSeq_origial.head()
x_labels= rnaSeq_origial["Status"].value_counts()


In [3]:
rnaSeq_origial["Age_C"] = np.where(rnaSeq_origial["Age"] > 65, "Old", "Young")
rnaSeq_origial["Class"]=rnaSeq_origial["Status"].astype(str) +" "+ rnaSeq_origial["Age_C"].astype(str)
remove_youn = rnaSeq_origial[rnaSeq_origial["Class"]!="Healthy Old"]
#remove_youn = rnaSeq_origial[rnaSeq_origial["Age_C"]!="Young"]

In [6]:
scorer = make_scorer(matthews_corrcoef)


In [7]:
seq_X_train, seq_X_test, seq_y_train, seq_y_test, seq_z_train, seq_z_test = train_test_split(remove_youn.drop(["Sample", "Status", "Age","Sex", "Age_C", "Class", "Experiment"], axis=1), remove_youn["Status"], remove_youn["Age"], test_size=0.3, random_state=42)


In [8]:
from sklearn.svm import SVC

# Create an SVM model
svm_model = SVC(kernel='sigmoid', C=1, gamma=0.1)

# Fit the model to the training data
svm_model.fit(seq_X_train, seq_y_train)

# Predict the labels for the test data
seq_y_pred = svm_model.predict(seq_X_test)
print(seq_y_test, seq_y_pred)

2     Sarcopenia
20       Healthy
13       Healthy
3        Healthy
22       Healthy
8        Healthy
29       Healthy
Name: Status, dtype: object ['Healthy' 'Healthy' 'Healthy' 'Healthy' 'Healthy' 'Healthy' 'Healthy']


In [9]:
seq_y_train

17       Healthy
5     Sarcopenia
6     Sarcopenia
25    Sarcopenia
18       Healthy
26       Healthy
24       Healthy
4     Sarcopenia
14       Healthy
31       Healthy
12    Sarcopenia
15    Sarcopenia
21       Healthy
28    Sarcopenia
11       Healthy
Name: Status, dtype: object

In [10]:
ridge_selection = f"/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/Results/master/feature_selection/RNAs_z_Combat_feature_selection.csv"
df = gtg.get_df(ridge_selection)# alpha=0.05
rnaseq_genes = df["Ensembl"]
len(rnaseq_genes)
rnaseq_genes = rnaseq_genes.to_list()

rnaseq_genes.append("Status")
eigth_bins_filteres = remove_youn[rnaseq_genes]


 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
 ** On entry to DLASCL parameter number  5 had an illegal value
 ** On entry to DLASCL parameter number  4 had an illegal value
0.1450134054998397


/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/getting_threshold_genes.py:21: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if gene_rank_df.iloc[0][1] == "coef":
/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/.venv/lib/python3.10/site-packages/pandas/core/indexes/base.py:945: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*new_inputs, **kwargs)
/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/.venv/lib/python3.10/site-packages/numpy/lib/polynomial.py:668: RuntimeWarning: invalid value encountered in divide
  lhs /= scale


In [11]:
seq_X_train, seq_X_test, seq_y_train, seq_y_test, seq_z_train, seq_z_test = train_test_split(eigth_bins_filteres.drop([ "Status"], axis=1), eigth_bins_filteres["Status"], eigth_bins_filteres, test_size=0.3, random_state=42)

# Create an SVM model
svm_model = SVC()

# Fit the model to the training data
svm_model.fit(seq_X_train, seq_y_train)

# Predict the labels for the test data
seq_y_pred = svm_model.predict(seq_X_test)
print(seq_y_test, seq_y_pred)

2     Sarcopenia
20       Healthy
13       Healthy
3        Healthy
22       Healthy
8        Healthy
29       Healthy
Name: Status, dtype: object ['Healthy' 'Healthy' 'Healthy' 'Healthy' 'Healthy' 'Healthy' 'Healthy']


In [12]:
from sklearn.model_selection import cross_val_score
from sklearn.svm import SVC

# Create an SVM model
svm_model = SVC(kernel='linear', C=100, gamma=0.00001)

# Perform cross-validation
cv_scores = cross_val_score(svm_model, seq_X_train, seq_y_train, cv=6,scoring=scorer)  # Change cv as needed

# Print cross-validation scores
print("Cross-Validation Scores:", cv_scores)
print("Mean CV Score:", cv_scores.mean())

# Fit the model to the entire training data
svm_model.fit(seq_X_train, seq_y_train)

# Predict the labels for the test data
seq_y_pred = svm_model.predict(seq_X_test)
print("Test Labels:", seq_y_test.values)
print("Predicted Labels:", seq_y_pred)


Cross-Validation Scores: [ 0.  0.  0.  0.  0. -1.]
Mean CV Score: -0.16666666666666666
Test Labels: ['Sarcopenia' 'Healthy' 'Healthy' 'Healthy' 'Healthy' 'Healthy' 'Healthy']
Predicted Labels: ['Sarcopenia' 'Healthy' 'Healthy' 'Healthy' 'Healthy' 'Healthy' 'Healthy']


In [13]:
seq_X_train.shape

(15, 458)

In [14]:
train_df = pd.DataFrame(seq_X_train)
train_df["Age"] = seq_y_train
smote = fs.get_feature_SMOTE(train_df, is_categorical=True)
 

Class distribution before SMOTE: Counter({'Healthy': 8, 'Sarcopenia': 7})
Class distribution after SMOTE: Counter({'Healthy': 8, 'Sarcopenia': 8})


In [15]:
import numpy as np
train_df = smote[0]
# add noise to train_df
# Generate random noise with the same shape as train_df
noise = np.random.normal(-0.001, 0.001, size=train_df.shape)

# Add the noise to train_df
noisy_train_df = train_df + noise

train_df["Status"] = smote[1]


/tmp/ipykernel_96920/3470635734.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_df["Status"] = smote[1]


In [16]:
svm_model = SVC()

X = train_df.drop(["Status"], axis=1)
y = train_df["Status"]
# Perform cross-validation
svm_model = SVC(kernel='sigmoid', C=100, gamma=0.00001)

# Perform cross-validation
cv_scores = cross_val_score(svm_model, X, y, cv=6,scoring=scorer)  # Change cv as needed

# Print cross-validation scores
print("Cross-Validation Scores:", cv_scores)
print("Mean CV Score:", cv_scores.mean())

# Fit the model to the entire training data
svm_model.fit(X, y)

# Predict the labels for the test data
seq_y_pred = svm_model.predict(seq_X_test)
print("Test Labels:", seq_y_test.values)
print("Predicted Labels:", seq_y_pred)

Cross-Validation Scores: [0. 0. 0. 0. 0. 0.]
Mean CV Score: 0.0
Test Labels: ['Sarcopenia' 'Healthy' 'Healthy' 'Healthy' 'Healthy' 'Healthy' 'Healthy']
Predicted Labels: ['Healthy' 'Sarcopenia' 'Sarcopenia' 'Sarcopenia' 'Healthy' 'Healthy'
 'Healthy']


In [17]:
top_35_genes_age = ["ENSG00000101892.11",
"ENSG00000277654.5",
"ENSG00000127241.16",
"ENSG00000171055.14",
"ENSG00000170873.18",
"ENSG00000137700.17",
"ENSG00000177548.12",
"ENSG00000123095.5",
"ENSG00000111640.14",
"ENSG00000127184.12",
"ENSG00000089101.17",
"ENSG00000101745.16",
"ENSG00000082397.17",
"ENSG00000178053.17",
"ENSG00000227097.5",
"ENSG00000185760.15",
"ENSG00000145675.14",
"ENSG00000124370.10",
"ENSG00000130935.9",
"ENSG00000185551.14",
"ENSG00000270388.1",
"ENSG00000131378.13",
"ENSG00000222427.1",
"ENSG00000150722.10",
"ENSG00000112562.18",
"ENSG00000169083.16",
"ENSG00000133315.10",
"ENSG00000177508.11",
"ENSG00000182108.10",
"ENSG00000199455.1",
"ENSG00000085231.13",
"ENSG00000238391.1",
"ENSG00000112031.15",
"ENSG00000188176.11",
"ENSG00000069275.12",
"Status"
]


In [18]:
eigth_bins_filteres=remove_youn.drop(columns=["Sample", "Age","Sex", "Age_C", "Class", "Experiment"])
seq_X_train, seq_X_test, seq_y_train, seq_y_test, seq_z_train, seq_z_test = train_test_split(eigth_bins_filteres.drop([ "Status"], axis=1), eigth_bins_filteres["Status"], eigth_bins_filteres, test_size=0.3, random_state=7)

from sklearn.metrics import make_scorer, matthews_corrcoef
from sklearn.model_selection import cross_val_score
from sklearn.svm import SVC

# Define MCC as a scoring function

# Create an SVM model
svm_model = SVC()
svm_model = SVC(kernel='linear', C=10000, gamma=0.00001)

# Perform cross-validation
cv_scores = cross_val_score(svm_model, seq_X_train, seq_y_train, cv=6,scoring=scorer)  # Change cv as needed

# Print cross-validation scores
print("Cross-Validation Scores:", cv_scores)
print("Mean CV Score:", cv_scores.mean())


# Fit the model to the entire training data
svm_model.fit(seq_X_train, seq_y_train)

# Predict the labels for the test data
seq_y_pred = svm_model.predict(seq_X_test)
print("Test Labels:", seq_y_test.values)
print("Predicted Labels:", seq_y_pred)
mcc = matthews_corrcoef(seq_y_test, seq_y_pred)
mcc

Cross-Validation Scores: [ 0.  -0.5 -0.5  0.   1.   0. ]
Mean CV Score: 0.0
Test Labels: ['Healthy' 'Healthy' 'Sarcopenia' 'Healthy' 'Sarcopenia' 'Healthy'
 'Healthy']
Predicted Labels: ['Sarcopenia' 'Healthy' 'Healthy' 'Healthy' 'Sarcopenia' 'Sarcopenia'
 'Sarcopenia']


-0.09128709291752768

In [19]:
coefs = svm_model.coef_
coefs_df = pd.DataFrame(coefs, columns=seq_X_train.columns)
coefs_df = coefs_df.T
coefs_df.columns = ["Coef"]
coefs_df["abs"] = coefs_df["Coef"].abs()
coefs_df = coefs_df.sort_values(by="abs", ascending=False)
coefs_df.head(50)

,Coef,abs
ENSG00000198467.14,-1.064920,1.064920
ENSG00000212907.2,0.631224,0.631224
ENSG00000183091.19,0.598397,0.598397
ENSG00000143549.19,-0.500243,0.500243
ENSG00000060138.12,0.382575,0.382575
ENSG00000142541.16,0.345249,0.345249
ENSG00000147689.16,0.337884,0.337884
ENSG00000138326.19,0.336754,0.336754
ENSG00000256618.2,0.333637,0.333637
ENSG00000108515.17,-0.301723,0.301723


In [20]:
eigth_bins_filteres= eigth_bins_filteres[top_35_genes_age]
#eigth_bins_filteres=remove_youn.drop(columns=["Sample", "Age","Sex", "Age_C", "Class", "Experiment"])
seq_X_train, seq_X_test, seq_y_train, seq_y_test, seq_z_train, seq_z_test = train_test_split(eigth_bins_filteres.drop([ "Status"], axis=1), eigth_bins_filteres["Status"], eigth_bins_filteres, test_size=0.2, random_state=30)

from sklearn.metrics import make_scorer, matthews_corrcoef
from sklearn.model_selection import cross_val_score
from sklearn.svm import SVC

# Define MCC as a scoring function

# Create an SVM model
svm_model = SVC()
svm_model = SVC(kernel='linear', C=10000, gamma=0.00001)

# Perform cross-validation
cv_scores = cross_val_score(svm_model, seq_X_train, seq_y_train, cv=6,scoring=scorer)  # Change cv as needed

# Print cross-validation scores
print("Cross-Validation Scores:", cv_scores)
print("Mean CV Score:", cv_scores.mean())


# Fit the model to the entire training data
svm_model.fit(seq_X_train, seq_y_train)

# Predict the labels for the test data
seq_y_pred = svm_model.predict(seq_X_test)
print("Test Labels:", seq_y_test.values)
print("Predicted Labels:", seq_y_pred)
mcc = matthews_corrcoef(seq_y_test, seq_y_pred)
mcc

Cross-Validation Scores: [ 0.   0.  -1.  -0.5  0.   0. ]
Mean CV Score: -0.25
Test Labels: ['Healthy' 'Sarcopenia' 'Sarcopenia' 'Healthy' 'Sarcopenia']
Predicted Labels: ['Sarcopenia' 'Sarcopenia' 'Healthy' 'Sarcopenia' 'Sarcopenia']


/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/.venv/lib/python3.10/site-packages/sklearn/model_selection/_split.py:737: UserWarning: The least populated class in y has only 5 members, which is less than n_splits=6.
  warnings.warn(


-0.4082482904638631

In [27]:
#eigth_bins_filteres= eigth_bins_filteres[top_35_genes_age]
eigth_bins_filteres=remove_youn.drop(columns=["Sample", "Age","Sex", "Age_C", "Class", "Experiment"])
seq_X_train, seq_X_test, seq_y_train, seq_y_test, seq_z_train, seq_z_test = train_test_split(eigth_bins_filteres.drop([ "Status"], axis=1), eigth_bins_filteres["Status"], eigth_bins_filteres, test_size=0.3, random_state=30)

from sklearn.metrics import make_scorer, matthews_corrcoef
from sklearn.model_selection import cross_val_score
from sklearn.svm import SVC

# Define MCC as a scoring function

# Create an SVM model
svm_model = SVC()
svm_model = SVC(kernel='linear', C=10000, gamma=0.00001)

# Perform cross-validation
cv_scores = cross_val_score(svm_model, seq_X_train, seq_y_train, cv=6,scoring=scorer)  # Change cv as needed

# Print cross-validation scores
print("Cross-Validation Scores:", cv_scores)
print("Mean CV Score:", cv_scores.mean())


# Fit the model to the entire training data
svm_model.fit(seq_X_train, seq_y_train)

# Predict the labels for the test data
seq_y_pred = svm_model.predict(seq_X_test)
print("Test Labels:", seq_y_test.values)
print("Predicted Labels:", seq_y_pred)

/home/karen/Documents/GitHub/Identify_muscle_age_genes/deep_learning/.venv/lib/python3.10/site-packages/sklearn/model_selection/_split.py:737: UserWarning: The least populated class in y has only 5 members, which is less than n_splits=6.
  warnings.warn(


Cross-Validation Scores: [-0.5 -0.5  0.5  0.   0.   1. ]
Mean CV Score: 0.08333333333333333
Test Labels: ['Healthy' 'Sarcopenia' 'Sarcopenia' 'Healthy' 'Sarcopenia' 'Healthy'
 'Healthy']
Predicted Labels: ['Healthy' 'Healthy' 'Healthy' 'Healthy' 'Sarcopenia' 'Healthy' 'Healthy']


In [28]:
mcc = matthews_corrcoef(seq_y_test, seq_y_pred)
mcc

0.47140452079103173

In [23]:
# get coefficientes of sigmoid svm model
coefs = svm_model.coef_
coefs_df = pd.DataFrame(coefs, columns=seq_X_train.columns)


In [24]:
coefs_df = coefs_df.T
coefs_df.columns = ["Coef"]
coefs_df["abs"] = coefs_df["Coef"].abs()
coefs_df = coefs_df.sort_values(by="abs", ascending=False)
coefs_df.head(50)

,Coef,abs
ENSG00000198467.14,-0.715241,0.715241
ENSG00000183091.19,0.603290,0.603290
ENSG00000182149.20,-0.436526,0.436526
ENSG00000251705.1,0.400649,0.400649
ENSG00000256618.2,0.388432,0.388432
ENSG00000198763.3,-0.320356,0.320356
ENSG00000180209.11,-0.309832,0.309832
ENSG00000163092.19,-0.309301,0.309301
ENSG00000162614.18,-0.267846,0.267846
ENSG00000199480.1,0.265718,0.265718


In [25]:
#eigth_bins_filteres=remove_youn.drop(columns=["Sample", "Age","Sex", "Age_C", "Class", "Experiment"])
seq_X_train, seq_X_test, seq_y_train, seq_y_test, seq_z_train, seq_z_test = train_test_split(eigth_bins_filteres.drop([ "Status"], axis=1), eigth_bins_filteres["Status"], eigth_bins_filteres, test_size=0.3, random_state=30)


# Define MCC as a scoring function

# Create an SVM model
svm_model = SVC()
svm_model = SVC(kernel='sigmoid', C=1000000, gamma=0.00001)

# Perform cross-validation
cv_scores = cross_val_score(svm_model, seq_X_train, seq_y_train, cv=5,scoring=scorer)  # Change cv as needed

# Print cross-validation scores
print("Cross-Validation Scores:", cv_scores)
print("Mean CV Score:", cv_scores.mean())


# Fit the model to the entire training data
svm_model.fit(seq_X_train, seq_y_train)

# Predict the labels for the test data
seq_y_pred = svm_model.predict(seq_X_test)
print("Test Labels:", seq_y_test.values)
print("Predicted Labels:", seq_y_pred)

Cross-Validation Scores: [-0.5  0.5  0.5  0.5  1. ]
Mean CV Score: 0.4
Test Labels: ['Healthy' 'Sarcopenia' 'Sarcopenia' 'Healthy' 'Sarcopenia' 'Healthy'
 'Healthy']
Predicted Labels: ['Healthy' 'Healthy' 'Healthy' 'Healthy' 'Sarcopenia' 'Healthy' 'Healthy']


In [26]:
mcc = matthews_corrcoef(seq_y_test, seq_y_pred)
mcc

0.47140452079103173